# Problem Class 2: Representations and Neural Networks on Handwritten Digits

In this tutorial we use one small image dataset throughout: handwritten digits. You will:

1. use **PCA** as a dimension-reduction baseline;
2. train a simple **multilayer perceptron (MLP)** in PyTorch;
3. inspect training curves, mistakes, and tensor shapes;
4. train a tiny **convolutional neural network (CNN)**.

The notebook is designed to run on a laptop CPU. No GPU or internet download is required.

## Instructions

- Run cells from top to bottom.
- Work through the **core analysis** in Parts 1--5 and the final comparison. (Part 0 is just setup, which you always run.)
- Use the **validation set** for model choices during the tutorial.
- Save the **test set** for the end, after choices have been made.
- Do not worry if PCA + logistic regression beats the neural networks on this small dataset. Simple baselines are valuable and can work well!

## Part 0 — Setup

We import the packages used in the tutorial and fix random seeds so that results are fairly reproducible.

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

# Small CPU-only tutorial: limiting threads avoids slowdowns on some shared machines.
torch.set_num_threads(1)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print("NumPy version:", np.__version__)
print("PyTorch version:", torch.__version__)

In [ ]:
def plot_digit_grid(images, labels=None, predictions=None, n=10):
    # Plot a small row of 8x8 digit images.
    n = min(n, len(images))
    fig, axes = plt.subplots(1, n, figsize=(1.2 * n, 1.4))
    if n == 1:
        axes = [axes]
    for i, ax in enumerate(axes):
        ax.imshow(images[i], cmap="gray_r", vmin=0, vmax=1)
        ax.axis("off")
        title = ""
        if labels is not None:
            title += f"y={labels[i]}"
        if predictions is not None:
            title += "\n" + f"ŷ={predictions[i]}"
        ax.set_title(title, fontsize=9)
    plt.tight_layout()
    plt.show()


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def layer_parameter_table(model):
    rows = []
    for name, module in model.named_modules():
        if isinstance(module, (nn.Linear, nn.Conv2d)):
            params = sum(p.numel() for p in module.parameters() if p.requires_grad)
            rows.append({"layer": name, "type": module.__class__.__name__, "parameters": params})
    return pd.DataFrame(rows)

## Part 1 — Load and inspect the digits dataset

Each digit is an $8\times 8$ greyscale image. We will use two views of the same data:

- `images`: shape `(n_samples, 8, 8)` for plotting and CNNs;
- `X`: shape `(n_samples, 64)` for PCA and the MLP.

Pixel values are divided by 16 so that they lie between 0 and 1. Since all features are pixels on the same scale, we will not standardise each pixel to unit variance before PCA in this tutorial.

In [ ]:
digits = load_digits()
images = digits.images.astype(np.float32) / 16.0
X = digits.data.astype(np.float32) / 16.0
y = digits.target.astype(np.int64)

print("images shape:", images.shape)
print("X shape:", X.shape)
print("y shape:", y.shape)
print("classes:", np.unique(y))

plot_digit_grid(images, y, n=10)

### Train/validation/test split

We use a 60/20/20 split. The validation set is used during the tutorial to compare choices such as PCA dimension or hidden-layer width. The test set is held back until the final comparison.

In [ ]:
X_trainval, X_test, y_trainval, y_test, img_trainval, img_test = train_test_split(
    X, y, images,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=y,
)

X_train, X_val, y_train, y_val, img_train, img_val = train_test_split(
    X_trainval, y_trainval, img_trainval,
    test_size=0.25,   # 0.25 of 80% = 20% of full data
    random_state=RANDOM_SEED,
    stratify=y_trainval,
)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

## Part 2 — PCA baseline

PCA gives a low-dimensional representation of the image vectors. We fit PCA **only on the training set**, then train a logistic regression classifier on the PCA scores.

For generic tabular data, standardisation before PCA is often important. For these digit images, the pixel values are already on a common 0--1 scale, so we use PCA directly on the rescaled pixels. PCA still centres the data internally.

**Experiment:** change the number of PCA components. How many components are enough for good validation accuracy?

In [ ]:
components_to_try = [2, 5, 10, 20, 40]

pca_results = []
for k in components_to_try:
    # PCA is fit on the training set only. Logistic regression is trained on the resulting PCA scores.
    pca_clf = make_pipeline(
        PCA(n_components=k, random_state=RANDOM_SEED),
        LogisticRegression(max_iter=3000),
    )

    pca_clf.fit(X_train, y_train)
    val_acc = pca_clf.score(X_val, y_val)
    pca_results.append({"PCA components": k, "validation accuracy": val_acc})

pca_results_df = pd.DataFrame(pca_results)
pca_results_df

In [ ]:
# Plot validation accuracy as a function of PCA dimension.
plt.figure(figsize=(5.5, 3.5))
plt.plot(pca_results_df["PCA components"], pca_results_df["validation accuracy"], marker="o")
plt.xlabel("number of PCA components")
plt.ylabel("validation accuracy")
plt.title("PCA dimension vs validation accuracy")
plt.ylim(0, 1.02)
plt.grid(alpha=0.3)
plt.show()

### Visualise the first two principal components

Two components can be useful for visualisation, but they may not be enough for classification.

In [ ]:
pca2 = PCA(n_components=2, random_state=RANDOM_SEED)
Z_train_2d = pca2.fit_transform(X_train)

plt.figure(figsize=(6, 4.8))
scatter = plt.scatter(Z_train_2d[:, 0], Z_train_2d[:, 1], c=y_train, s=14, alpha=0.8, cmap="tab10")
plt.xlabel("PC1 score")
plt.ylabel("PC2 score")
plt.title("Training digits projected onto first two PCs")
plt.colorbar(scatter, ticks=range(10), label="digit")
plt.tight_layout()
plt.show()

**Question:** Why might two PCA components be good for visualisation but not enough for classification?

## Part 3 — A first MLP in PyTorch

The MLP sees each image as a 64-dimensional vector:

$$
64 \longrightarrow 32 \longrightarrow 10.
$$

The final output has 10 raw class scores, also called **logits**. We will use these to compute the cross entropy loss in PyTorch.

**What `nn.CrossEntropyLoss()` expects:**

- **Predictions:** raw **logits** of shape `(batch_size, 10)`, *not* softmax probabilities. The loss applies `log_softmax` internally, so if you softmax first you apply it twice and weaken the gradient.
- **Targets:** integer **class indices** of dtype `torch.long` and shape `(batch_size,)`, *not* one-hot vectors and *not* floats. Internally the loss is `log_softmax` followed by negative log-likelihood, so it uses each integer to index into the log-probabilities.

In short, `loss_fn(logits, y)` wants a `(N, 10)` float tensor and a `(N,)` long tensor. The next cell builds these.

In [ ]:
# Convert NumPy arrays to PyTorch tensors.
# Inputs are float32 of shape (N, 64); labels are torch.long class indices
# of shape (N,) - exactly what nn.CrossEntropyLoss expects (see the cell above).
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

batch_size = 64
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=batch_size, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=batch_size, shuffle=False)

In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(64, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 10),
        )

    def forward(self, x):
        return self.net(x)

model = MLP(hidden_dim=32)
print(model)
print("Trainable parameters:", count_parameters(model))
layer_parameter_table(model)

### Shape check

Before training, always check tensor shapes.

**Predict first:** before running the next cell, write down the shape you expect for each of `X_batch`, `y_batch`, and `model(X_batch)`. Then run it and check. (Reminder: there are 10 classes and the batch size is 64.)

In [ ]:
X_batch, y_batch = next(iter(train_loader))
logits = model(X_batch)

print("X_batch shape:", X_batch.shape)
print("y_batch shape:", y_batch.shape)
print("model(X_batch) shape:", logits.shape)
print("first row of logits:", logits[0].detach().numpy().round(3))

### Training and evaluation functions

This is the main part of the tutorial. Here are the full details for implementing the training loop in PyTorch.

**Run once per batch, in this order:**

1. `optimizer.zero_grad()` — reset gradients to zero. PyTorch *accumulates* gradients into each parameter's `.grad`, so if you forget this they pile up across batches and training misbehaves. (This is the most common bug.)
2. `logits = model(X_batch)` — the **forward pass**: compute predictions.
3. `loss = loss_fn(logits, y_batch)` — compute the loss to measure how wrong the predictions are.
4. `loss.backward()` — the **backward pass**: compute the gradient of the loss with respect to every parameter and store it in each parameter's `.grad`. This **does not** change any weights.
5. `optimizer.step()` — apply **one update step**, nudging each weight using its `.grad`. This is the line that actually changes the model. It works because we built the optimizer with `model.parameters()`, so it knows which tensors to update.

Steps 4 and 5 are the two `TODO` lines you will fill in. Note the division of labour: `backward()` *computes* gradients, `step()` *applies* them. Both are needed to complete the training successfully.

**A few other PyTorch mechanics used below:**

- `logits.argmax(dim=1)` turns a `(batch, 10)` tensor of scores into a `(batch,)` tensor of predicted classes — the index of the largest score in each row.
- `model.train()` vs `model.eval()` switch the model between training and evaluation behaviour. They make no difference for this MLP (no dropout or batch-normalisation), but they matter the moment a model has those layers, so it is a good habit.
- `with torch.no_grad():` tells PyTorch not to track gradients during evaluation. We are not training there, so this saves time and memory.
- The `DataLoader` uses `shuffle=True` for training (batches differ each epoch) but `shuffle=False` for validation and test (order does not matter when we are only measuring).

In [ ]:
def train_one_epoch(model, loader, optimizer, loss_fn):
    model.train()
    total_loss, total_correct, total = 0.0, 0, 0

    for X_batch, y_batch in loader:
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = loss_fn(logits, y_batch)

        # TODO 1: compute the gradients via backpropagation.
        # This fills each parameter's .grad; it does NOT change any weights.
        # (See the training-loop instructions in the cell above.)
        raise NotImplementedError("TODO: compute gradients")

        # TODO 2: apply one optimizer update step using those gradients.
        # This is the line that actually changes the model's weights.
        raise NotImplementedError("TODO: update parameters")

        total_loss += loss.item() * X_batch.size(0)
        total_correct += (logits.argmax(dim=1) == y_batch).sum().item()
        total += X_batch.size(0)

    return total_loss / total, total_correct / total

In [ ]:
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss, total_correct, total = 0.0, 0, 0
    all_preds, all_targets = [], []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            logits = model(X_batch)
            loss = loss_fn(logits, y_batch)
            preds = logits.argmax(dim=1)

            total_loss += loss.item() * X_batch.size(0)
            total_correct += (preds == y_batch).sum().item()
            total += X_batch.size(0)
            all_preds.append(preds.numpy())
            all_targets.append(y_batch.numpy())

    return {
        "loss": total_loss / total,
        "accuracy": total_correct / total,
        "preds": np.concatenate(all_preds),
        "targets": np.concatenate(all_targets),
    }


def train_model(model, train_loader, val_loader, epochs=25, lr=1e-3):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, loss_fn)
        val_metrics = evaluate(model, val_loader, loss_fn)
        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["accuracy"],
        })

        if epoch == 1 or epoch % 5 == 0 or epoch == epochs:
            print(
                f"epoch {epoch:02d} | "
                f"train loss {train_loss:.3f}, acc {train_acc:.3f} | "
                f"val loss {val_metrics['loss']:.3f}, acc {val_metrics['accuracy']:.3f}"
            )

    return pd.DataFrame(history)

### Train the MLP

**Exercise - can you beat the baseline?** After your first successful run, try changing `hidden_dim` (e.g. `8`, `32`, `64`, `128`) or `lr` (e.g. `1e-4`, `1e-3`, `1e-2`) and compare *validation* accuracy. Your target to beat is the best PCA + logistic-regression result from Part 2. Track what helps - and do not peek at the test set while tuning.

In [ ]:
torch.manual_seed(RANDOM_SEED)
mlp = MLP(hidden_dim=32)
mlp_history = train_model(mlp, train_loader, val_loader, epochs=25, lr=1e-3)
mlp_history.tail()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
ax[0].plot(mlp_history["epoch"], mlp_history["train_loss"], label="train")
ax[0].plot(mlp_history["epoch"], mlp_history["val_loss"], label="validation")
ax[0].set_xlabel("epoch")
ax[0].set_ylabel("loss")
ax[0].set_title("Loss curves")
ax[0].legend()
ax[0].grid(alpha=0.3)

ax[1].plot(mlp_history["epoch"], mlp_history["train_acc"], label="train")
ax[1].plot(mlp_history["epoch"], mlp_history["val_acc"], label="validation")
ax[1].set_xlabel("epoch")
ax[1].set_ylabel("accuracy")
ax[1].set_ylim(0, 1.02)
ax[1].set_title("Accuracy curves")
ax[1].legend()
ax[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Part 4 — Validation diagnostics

We now inspect predictions on the validation set. Look especially at the mistakes: are they visually understandable?

**Predict first:** which two digits do you think the model confuses most often? (Common culprits on this dataset are 4 vs 9, 3 vs 8, and 5 vs 8.) Check your guess against the confusion matrix below.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
val_metrics = evaluate(mlp, val_loader, loss_fn)
print("MLP validation accuracy:", val_metrics["accuracy"])

cm = confusion_matrix(val_metrics["targets"], val_metrics["preds"])
ConfusionMatrixDisplay(cm, display_labels=np.arange(10)).plot(values_format="d")
plt.title("MLP validation confusion matrix")
plt.show()

In [ ]:
# Show a few mistakes on the validation set.
val_preds = val_metrics["preds"]
wrong = np.where(val_preds != y_val)[0]
print("Number of validation mistakes:", len(wrong))

if len(wrong) > 0:
    idx = wrong[:10]
    plot_digit_grid(img_val[idx], labels=y_val[idx], predictions=val_preds[idx], n=len(idx))
else:
    print("No validation mistakes to show!")

## Part 5 — A convolutional neural network (CNN)

A CNN sees each image as a tensor of shape `(channels, height, width)` rather than as a flattened vector. Here the images are greyscale, so there is one channel:

$$
(N,64) \quad \longrightarrow \quad (N,1,8,8).
$$

In [ ]:
# Reshape X_train, X_val and X_test into image tensors of shape (N, 1, 8, 8).
# PyTorch CNNs expect image batches in (batch, channels, height, width) format.
X_train_img = X_train.reshape(-1, 1, 8, 8).astype(np.float32)
X_val_img = X_val.reshape(-1, 1, 8, 8).astype(np.float32)
X_test_img = X_test.reshape(-1, 1, 8, 8).astype(np.float32)

train_img_loader = DataLoader(
    TensorDataset(torch.tensor(X_train_img), y_train_t),
    batch_size=batch_size,
    shuffle=True,
)
val_img_loader = DataLoader(
    TensorDataset(torch.tensor(X_val_img), y_val_t),
    batch_size=batch_size,
    shuffle=False,
)
test_img_loader = DataLoader(
    TensorDataset(torch.tensor(X_test_img), y_test_t),
    batch_size=batch_size,
    shuffle=False,
)

print("CNN input tensor shape:", torch.tensor(X_train_img).shape)

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Linear(16 * 2 * 2, 10)

    def forward(self, x):
        h = self.features(x)
        h = torch.flatten(h, start_dim=1)
        return self.classifier(h)

cnn = TinyCNN()
print(cnn)
print("Trainable parameters:", count_parameters(cnn))
layer_parameter_table(cnn)

### CNN shape tracing

This cell applies each feature-extraction layer one at a time.

In [ ]:
X_img_batch, y_img_batch = next(iter(train_img_loader))
print("input:", X_img_batch[:4].shape)

h = X_img_batch[:4]
for layer in cnn.features:
    h = layer(h)
    print(f"{layer.__class__.__name__:10s}", h.shape)

logits = cnn(X_img_batch[:4])
print("logits:", logits.shape)

### Train the tiny CNN

The dataset is very small, so the CNN may not dramatically outperform the MLP. The main goal is to understand shapes and the training loop.

In [ ]:
torch.manual_seed(RANDOM_SEED)
cnn = TinyCNN()
cnn_history = train_model(cnn, train_img_loader, val_img_loader, epochs=20, lr=1e-3)
cnn_history.tail()

In [ ]:
plt.figure(figsize=(5.5, 3.5))
plt.plot(cnn_history["epoch"], cnn_history["train_acc"], label="CNN train")
plt.plot(cnn_history["epoch"], cnn_history["val_acc"], label="CNN validation")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.ylim(0, 1.02)
plt.title("Tiny CNN accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Part 6 — Final comparison on the test set

Now that model choices have been made using the validation set, we can evaluate on the held-out test set.

**Predict first:** rank the three models - PCA + logistic regression, the MLP, and the tiny CNN - from most to least accurate on the test set. Were you right?

For speed and fairness in this short tutorial, we evaluate the neural networks exactly as trained on the training set, and we also fit the PCA baseline on the training set only. A more formal final workflow would retrain the selected model on training + validation before testing.

Do not be surprised if the PCA/logistic-regression baseline is very strong on this small dataset. Strong simple baselines are useful, and neural networks do not automatically win on every small problem.

In [ ]:
# Choose PCA dimension using validation accuracy.
best_k = int(pca_results_df.sort_values("validation accuracy", ascending=False).iloc[0]["PCA components"])
print("Best PCA dimension from validation:", best_k)

# For a fair short-session comparison, fit the PCA baseline on the training set only.
final_pca_clf = make_pipeline(
    PCA(n_components=best_k, random_state=RANDOM_SEED),
    LogisticRegression(max_iter=3000),
)
final_pca_clf.fit(X_train, y_train)
pca_test_acc = final_pca_clf.score(X_test, y_test)

mlp_test = evaluate(mlp, test_loader, nn.CrossEntropyLoss())

rows = [
    {"model": f"PCA({best_k}) + logistic regression", "test accuracy": pca_test_acc},
    {"model": "MLP", "test accuracy": mlp_test["accuracy"]},
]

cnn_test = evaluate(cnn, test_img_loader, nn.CrossEntropyLoss())

rows.append({"model": "Tiny CNN", "test accuracy": cnn_test["accuracy"]})

comparison = pd.DataFrame(rows)
comparison